In [1]:
import numpy as np
from sklearn.pipeline import make_pipeline , Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder , StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

In [2]:
import seaborn as sns
import numpy as np

In [3]:
df = sns.load_dataset('tips')

In [4]:
df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [5]:
df.shape

(244, 7)

In [6]:
X = df.drop('total_bill',axis=1)
y = df['total_bill']

In [7]:
from sklearn.model_selection import train_test_split
X_train , X_test , y_train , y_test = train_test_split(X,y,test_size=0.2)

In [8]:
numerical_steps = [('impute_mean',SimpleImputer(missing_values=np.nan,strategy='mean')),
         ('scaler',StandardScaler())]

In [9]:
numerical_pipeline = Pipeline(steps=numerical_steps)

In [10]:
categorical_steps = [('imputation_const',SimpleImputer(fill_value='missing',strategy='constant')),
                     ('encoding',OneHotEncoder(handle_unknown='ignore'))]

In [11]:
categorical_pipeline = Pipeline(steps=categorical_steps)

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   total_bill  244 non-null    float64 
 1   tip         244 non-null    float64 
 2   sex         244 non-null    category
 3   smoker      244 non-null    category
 4   day         244 non-null    category
 5   time        244 non-null    category
 6   size        244 non-null    int64   
dtypes: category(4), float64(2), int64(1)
memory usage: 7.4 KB


In [13]:
cat_cols = X.select_dtypes('category')
num_cols = ['tip','size']

In [14]:
preprocessor = ColumnTransformer(
    [
        ('categorical',categorical_pipeline,['sex','smoker','day','time']),
        ('numerical',numerical_pipeline,['tip','size'])
    ]
)

In [15]:
final_steps = [('preprocessor',preprocessor),
               ('regressor',RandomForestRegressor())]

In [16]:
pipe = Pipeline(steps=final_steps)

In [17]:
pipe

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  Pipeline(steps=[('imputation_const',
                                                                   SimpleImputer(fill_value='missing',
                                                                                 strategy='constant')),
                                                                  ('encoding',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['sex', 'smoker', 'day',
                                                   'time']),
                                                 ('numerical',
                                                  Pipeline(steps=[('impute_mean',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['tip', 'size'])])),
                ('regressor', RandomForestRegressor())])

In [18]:
import warnings
warnings.filterwarnings('ignore')

In [19]:
param_grid = {
    'regressor__n_estimators':[200,500],
    'regressor__max_features':['auto','sqrt','log2'],
    'regressor__max_depth':[4,5,6,7,8]
}

In [20]:
grid_search = GridSearchCV(pipe,param_grid=param_grid,n_jobs=1)

In [21]:
grid_search.fit(X_train,y_train)

GridSearchCV(estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('categorical',
                                                                         Pipeline(steps=[('imputation_const',
                                                                                          SimpleImputer(fill_value='missing',
                                                                                                        strategy='constant')),
                                                                                         ('encoding',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         ['sex',
                                                                          'smoker',
                                                                          'day',
                                                                          'time']),
                                                                        ('numerical',
                                                                         Pipeline(steps=[('impute_mean',
                                                                                          SimpleImputer()),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['tip',
                                                                          'size'])])),
                                       ('regressor', RandomForestRegressor())]),
             n_jobs=1,
             param_grid={'regressor__max_depth': [4, 5, 6, 7, 8],
                         'regressor__max_features': ['auto', 'sqrt', 'log2'],
                         'regressor__n_estimators': [200, 500]})

In [45]:
grid_search.best_params_

{'regressor__max_depth': 7,
 'regressor__max_features': 'log2',
 'regressor__n_estimators': 200}

# Now update the pipeline's estimator which these hyperparameters

In [48]:
final_steps = [('preprocessor',preprocessor),
               ('regressor',RandomForestRegressor(max_depth=7,max_features='log2',n_estimators=200))]
pipe = Pipeline(steps=final_steps)

In [50]:
grid_search.fit(X_train,y_train)

GridSearchCV(estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('categorical',
                                                                         Pipeline(steps=[('imputation_const',
                                                                                          SimpleImputer(fill_value='missing',
                                                                                                        strategy='constant')),
                                                                                         ('encoding',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         ['sex',
                                                                          'smoker',
                                                                          'day',
                                                                          'time']),
                                                                        ('numerical',
                                                                         Pipeline(steps=[('impute_mean',
                                                                                          SimpleImputer()),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['tip',
                                                                          'size'])])),
                                       ('regressor', RandomForestRegressor())]),
             n_jobs=1,
             param_grid={'regressor__max_depth': [4, 5, 6, 7, 8],
                         'regressor__max_features': ['auto', 'sqrt', 'log2'],
                         'regressor__n_estimators': [200, 500]})

In [51]:
y_pred = grid_search.predict(X_test)

In [54]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [58]:
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f"R^2: {r2}, RMSE: {rmse}, MAE: {mae}")

R^2: 0.4555966616045032, RMSE: 6.722186580436993, MAE: 4.8389741162494255
